# Validation Test - Droplet in Equilibrium (transient simulations)

This notebook and the corresponding evaluation notebook (DropletInEquilibrium_transient_Evaluation.ipynb) are part of published results (cf. 7.2.1) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

MPI Rank 0: Value for OMP_PLACES: 
MPI Rank 0: Value for OMP_PROC_BIND: 
Number of CPUs in system: 24
not defined: OMP_NUM_THREADS
System reports 24 CPUs, 1 MPI ranks on current node.
Failed to determine user wish for number of threads; trying to use all! System reports 24 CPUs, will use all but 2 for BoSSS (22 total, 22 per MPI rank, MPI ranks on current node is 1).
Finally, setting number of OpenMP and Parallel Task Library threads to 22
CCP_AFFINITY not set
R0: reserved CPUs: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], C# reports mask FFFFFF
R0: reserved CPUs for this MPI rank: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], on entire SMP: [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23], disjount CPU affinities? False
MpiJobOwnsEntireComputer = True, RnkJobOwnsEntireComputer = True
R0: using CPUs [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23] for OpenMP.
R0: TPL thread pinning: False, OMP thread pinning: False
R0

Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 263
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 105
   at Submission#9.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [10]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [11]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [ ]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [ ]:
wmg.Init("XNSEpaper_Droplet", myBatch);

Project name is set to 'XNSEpaper_DropletInEquilibrium'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\XNSEpaper_DropletInEquilibrium'.


In [ ]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [15]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [16]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: XNSEpaper_DropletInEquilibrium	DropletInEquilibrium_meshStudy_mesh40*	10/13/2025 1:30:20 PM	d32e74f3...
#1: XNSEpaper_DropletInEquilibrium	DropletInEquilibrium_meshStudy_mesh20*	10/13/2025 1:30:08 PM	53f4f0eb...


## Grid Creation for Convergence Study

In [17]:
int[] Resolutions = new int[] { 20, 40, 60, 80 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

double L = 0.5; 

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"DropletInEquilibrium_{Res}";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = GenericBlas.Linspace(-L, L, Res + 1);
        double[] yNodes = xNodes;
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);
    
        grd.Name = GridName;
    
        grd.DefineEdgeTags(delegate (double[] X) {
            string ret = null;
            if (Math.Abs(X[0] + L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if (Math.Abs(X[0] - L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if (Math.Abs(X[1] + L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if (Math.Abs(X[1] - L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            return ret;
        });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Opening existing database 'C:\BoSSS-localJobs\XNSEpaper_DropletInEquilibrium'.
Grid Edge Tags changed.
An equivalent grid (a4c02c78-ebef-487f-a947-24d2e093654e) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (e24bc390-ec3b-4ad5-a00b-4b67e962db07) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (12ab1bf4-fd61-4198-88cd-bfd38f34014d) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (012cbaf6-5127-40f5-9dba-f5e26ae95548) is already present in the database -- the grid will not be saved.


## Initial Values

In [18]:
double r = 0.25;
r

0.25

In [19]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double dropRadius = 0.25;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] - 0.0).Pow2() + (X[1] - 0.0).Pow2()) - dropRadius; " + 
    "}");

## Physical Settings

In [20]:
double rho = 1e4;    
double mu = 1.0;
double sigma = 1.0;

double La = sigma * rho * 2 * r / mu.Pow2();
La

5000

In [21]:
double dt = 0.01;
double t_end = 125;

## Control settings

In [22]:
//int[] degS = new int[] { 2 };

In [26]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
for(int iGrd = 0; iGrd < Grids.Length; iGrd++) {

    if (iGrd > 1)   // we skip for now simualtions longer than one week
        continue;
    
    XNSE_Control C = new XNSE_Control();
    
    int pDeg = 2;   
    var grd  = Grids[iGrd];

    C.SetDGdegree(pDeg);
    
    // set grid
    C.SetGrid(grd);
    
    // initial conditions   
    C.AddInitialValue("Phi", PhiFunc);   
    
    C.PostprocessingModules.Add(new Dropletlike(){ SolverStage = 2, LogPeriod = 10});
    C.PostprocessingModules.Add(new EnergyLogging(){ SolverStage = 2, LogPeriod = 10});

    C.PhysicalParameters.rho_A = rho;
    C.PhysicalParameters.rho_B = rho;
    C.PhysicalParameters.mu_A  = mu;
    C.PhysicalParameters.mu_B  = mu;
    C.PhysicalParameters.Sigma = sigma;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;

    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.NonLinearSolver.MaxSolverIterations = 50;
    
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
  
    C.dtMin         = dt;
    C.dtMax         = dt;
    C.Endtime       = t_end;
    C.NoOfTimesteps = (int)(t_end/dt); 
    
    C.saveperiod = 125;
    
    C.SessionName = "DropletInEquilibrium_meshStudy_mesh" + Resolutions[iGrd];
    
    Controls.Add(C);
}

In [ ]:
foreach (var ctrl in Controls) {
    Console.WriteLine($"{ctrl.SessionName}");
}

DropletInEquilibrium_meshStudy_mesh20: dt = 0.01; saveperiod = 125; logperiod = 10
DropletInEquilibrium_meshStudy_mesh40: dt = 0.01; saveperiod = 125; logperiod = 10


In [29]:
int NoCtrls = Controls.Count();
NoCtrls

2

## Launch Jobs

In [23]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);
}

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletInEquilibrium_meshStudy_mesh20 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\XNSEpaper_DropletInEquilibrium-XNSE_Solver2025Oct13_132951.571695
copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletInEquilibrium_meshStudy_mesh40 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\XNSEpaper_DropletInEquilibrium-XNSE_Solver2025Oct13_133001.521669
copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.


## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.InProgress 
                                                                || job.Status == JobStatus.PendingInExecutionQueue
                                                                || job.Status == JobStatus.PreActivation).Count();
NoInProgress

In [ ]:
int maxDays = 7;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [ ]:
int NoFailed = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.FailedOrCanceled).Count();
NoFailed

In [ ]:
int NoSuccess = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.FinishedSuccessful).Count();
NoSuccess

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var ctrl in Controls) {
        var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(ctrl.SessionName));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {ctrl.SessionName} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {ctrl.SessionName}!"); // should not happen
            } 
        }
    }

}

In [ ]:
NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");

Error: (1,37): error CS0103: The name 'NoSuccess' does not exist in the current context